# 내구 실행 모드 비교

같은 지점에서 자식 프로세스를 종료했을 때 exit, async, sync가 남기는 체크포인트를 비교합니다.


## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.


In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11 langgraph-checkpoint-sqlite==3.0.3

Note: you may need to restart the kernel to use updated packages.


## 실험 함수 준비

자식 프로세스가 두 번째 노드에 진입하면 즉시 종료하고 SQLite 체크포인트를 직접 읽습니다.

내구 실행 모드는 완료된 단계의 체크포인트를 저장 호출과 어느 시점에 맞출지 정합니다. `sync`는 다음 단계로 진행하기 전에 저장 완료를 기다리고, `async`는 저장을 진행하면서 다음 작업을 이어 가며, `exit`는 실행이 끝날 때 상태를 기록합니다. 저장 시점이 다르면 갑작스러운 프로세스 종료 직전에 디스크에 남은 범위가 달라질 수 있습니다.

`kill_and_resume(mode)`는 같은 그래프와 입력을 자식 프로세스에서 실행하고, 두 번째 노드 진입을 알리는 ready 파일이 생기면 프로세스를 종료합니다. `read_saved`는 SQLite에서 최신 `channel_values`를 읽고, 같은 모드 이름을 `thread_id`로 사용해 새 프로세스로 재개합니다. ready 파일은 체크포인트 저장 완료 신호가 아니라 노드 진입 신호입니다.


In [2]:
# worker는 실제 종료할 자식 프로세스이고 run_dir는 모드별 SQLite 파일 위치입니다.
import sqlite3
import subprocess
import sys
import time
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

candidates = [Path.cwd(), *Path.cwd().parents]
lab = next((p / "labs/part6/ch23-persistence" for p in candidates if (p / "labs/part6/ch23-persistence").is_dir()), None)
if lab is None:
    lab = next(p for p in candidates if p.name == "ch23-persistence")
worker = lab / "durability_worker.py"
run_dir = lab / "data" / "durability"
run_dir.mkdir(parents=True, exist_ok=True)

## 체크포인트 읽기 함수

자식 프로세스가 종료된 뒤 같은 스레드의 최신 저장 상태를 읽습니다.


In [3]:
# read_saved는 프로세스 종료 뒤 thread_id에 연결된 최신 채널 값을 읽습니다.
def read_saved(db_path: Path, mode: str) -> dict | None:
    if not db_path.exists():
        return None
    with sqlite3.connect(db_path, check_same_thread=False) as conn:
        saved = SqliteSaver(conn).get_tuple({"configurable": {"thread_id": mode}})
    return None if saved is None else saved.checkpoint.get("channel_values", {})

## 중단 뒤 재개 함수

ready 파일은 두 번째 노드 진입 신호이고 deadline은 20초 대기 상한입니다.


In [4]:
# ready 파일을 두 번째 노드 진입 신호로 사용하고 deadline 20초 안에 자식 프로세스를 종료·재개합니다.
def kill_and_resume(mode: str) -> dict:
    db_path = run_dir / f"{mode}.sqlite"
    ready = run_dir / f"{mode}.ready"
    trace = ready.with_suffix(".trace")
    for path in [db_path, ready, trace]:
        path.unlink(missing_ok=True)
    process = subprocess.Popen([sys.executable, str(worker), str(db_path), str(ready), mode])
    deadline = time.time() + 20
    while not ready.exists() and time.time() < deadline:
        time.sleep(0.02)
    assert ready.exists(), f"{mode} worker did not enter second node"
    process.kill()
    process.wait(timeout=5)
    before = read_saved(db_path, mode)
    subprocess.run([sys.executable, str(worker), str(db_path), str(ready), mode, "resume"], check=True, timeout=20)
    after = read_saved(db_path, mode)
    calls = trace.read_text(encoding="utf-8").splitlines()
    return {"mode": mode, "before": before, "after": after, "calls": calls}

## 같은 중단 시점에서 세 모드 실행

실행 모드 외의 그래프, 입력, 종료 시점을 고정합니다.


In [5]:
# 그래프와 종료 시점을 고정하고 exit, async, sync만 바꾸어 세 번 실행합니다.
observations = [kill_and_resume(mode) for mode in ["exit", "async", "sync"]]
for row in observations:
    print(row)
assert observations[0]["before"] is None
assert "first" in observations[2]["before"].get("completed", [])
assert all(row["after"].get("completed") == ["first", "second"] for row in observations)

{'mode': 'exit', 'before': None, 'after': {'completed': ['first', 'second']}, 'calls': ['first', 'second', 'first', 'second']}
{'mode': 'async', 'before': {'completed': ['first'], 'branch:to:second': None}, 'after': {'completed': ['first', 'second']}, 'calls': ['first', 'second', 'second']}
{'mode': 'sync', 'before': {'completed': ['first'], 'branch:to:second': None}, 'after': {'completed': ['first', 'second']}, 'calls': ['first', 'second', 'second']}


## 손실 범위 요약

저장된 `completed`와 데이터베이스 크기를 함께 표시합니다.

결제 승인처럼 한 단계 손실도 허용하기 어려운 흐름은 저장 완료를 기다리는 모드가 필요할 수 있습니다. 대량 배치처럼 처리량이 우선이고 다시 계산할 수 있는 흐름은 비동기 저장의 지연 이점을 검토할 수 있습니다. 선택에는 저장소 지연과 재실행 비용을 함께 봐야 합니다.

특히 `async` 결과는 종료와 백그라운드 저장의 경합에 따라 달라질 수 있습니다. 이번 한 번의 관찰을 항상 같은 보장으로 읽으면 안 됩니다. 세 모드가 모두 재개 후 완료됐다는 결과와 종료 직전 저장 범위의 차이를 따로 해석합니다.


In [6]:
# 중단 전 저장값, 재개 후 완료값, 실제 노드 호출 순서를 모드별로 나란히 표시합니다.
for row in observations:
    before = None if row["before"] is None else row["before"].get("completed")
    print(f"{row['mode']:5} -> before={before}, after={row['after'].get('completed')}, calls={row['calls']}")
print("async는 저장 작업과 종료 시점의 경합에 따라 이번 관찰값과 달라질 수 있습니다.")

exit  -> before=None, after=['first', 'second'], calls=['first', 'second', 'first', 'second']
async -> before=['first'], after=['first', 'second'], calls=['first', 'second', 'second']
sync  -> before=['first'], after=['first', 'second'], calls=['first', 'second', 'second']
async는 저장 작업과 종료 시점의 경합에 따라 이번 관찰값과 달라질 수 있습니다.
